# **Object Detection: YOLOv11 on COCO Subset**

Notebook này train trên **subset của validation set** được chia thành 70/20/10 (training/validation/test).
Colab-ready variant saves checkpoints to Google Drive every epoch.

Goals:
- Train và compare **YOLO11n** và **YOLO11s** trên cùng subset.
- Chia validation set thành 70% training, 20% validation, 10% test.
- Lưu kết quả ở Google Drive.

- Resume sau Colab quota interruption bằng `last.pt`.

## **1. Chuẩn bị môi trường**
Chạy cell này để cài các thư viện cần thiết.

In [ ]:
%pip install -q ultralytics==8.3.0 pycocotools pandas seaborn pyyaml scikit-learn

In [ ]:
import json
import random
from pathlib import Path
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

sns.set_theme(style='whitegrid')
random.seed(42)
np.random.seed(42)

## **Google Drive setup for persistent checkpoints**
- On Colab: mount Drive and save runs under `MyDrive`.
- On local: fallback to current working directory.

In [ ]:
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/CO5085_DeepLearning_CV')
else:
    DRIVE_ROOT = Path.cwd()

print(f'IS_COLAB={IS_COLAB}')
print(f'DRIVE_ROOT={DRIVE_ROOT}')

## **2. Thiết lập bài toán và dữ liệu**
- Notebook này train trên **COCO subset** được tạo từ validation set.
- Subset được chia **70% training / 20% validation / 10% test**.
- Kết quả sẽ lưu ở Google Drive dưới `assignment2_yolov11_coco_subset`.

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

# ============ HELPER FUNCTIONS ============
def resolve_path(base_path, value):
    """Convert relative path to absolute if needed."""
    p = Path(value)
    return p if p.is_absolute() else (Path(base_path) / p)

def get_images_from_source(image_source, base_data_path=None):
    """Load images from .txt file or directory."""
    images = []
    image_source = Path(image_source)
    
    if image_source.is_file() and image_source.suffix.lower() == '.txt':
        with open(image_source, 'r', encoding='utf-8') as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                p = Path(s)
                if not p.is_absolute() and base_data_path:
                    p = Path(base_data_path) / p
                if p.exists() and p.is_file():
                    images.append(p)
    elif image_source.is_dir():
        images = sorted(
            list(image_source.glob('*.jpg')) +
            list(image_source.glob('*.jpeg')) +
            list(image_source.glob('*.png'))
        )
    
    return sorted(images)

def copy_subset_with_labels(image_list, src_label_dir, dest_img_dir, dest_label_dir, split_name=''):
    """Copy images and corresponding labels to destination."""
    copy_count = 0
    for img_path in image_list:
        # Copy image
        img_dest = dest_img_dir / img_path.name
        shutil.copy2(img_path, img_dest)
        
        # Copy label
        label_src = src_label_dir / (img_path.stem + '.txt')
        if label_src.exists():
            label_dest = dest_label_dir / (img_path.stem + '.txt')
            shutil.copy2(label_src, label_dest)
            copy_count += 1
    
    if split_name:
        print(f'  ✓ {split_name}: {copy_count}/{len(image_list)} labels copied')
    return copy_count

def create_subset_structure(subset_root, splits=['train', 'val', 'test']):
    """Create train/val/test folder structure."""
    dirs = {}
    for split in splits:
        img_dir = subset_root / split / 'images'
        lbl_dir = subset_root / split / 'labels'
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        dirs[split] = {'images': img_dir, 'labels': lbl_dir}
    return dirs

# ============ CONFIGURATION ============
USE_COCO128_DEBUG = False
DATA_YAML = 'coco_subset.yaml'
subset_yaml_path = DRIVE_ROOT / DATA_YAML

# Data split ratios: 70% train, 20% val, 10% test
SPLIT_RATIOS = {'train': 0.70, 'val': 0.20, 'test': 0.10}

# Setup project directory (persist on Drive)
PROJECT_DIR = DRIVE_ROOT / 'runs' / 'assignment2_yolov11_coco_subset'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# ============ CREATE SUBSET ============
if not subset_yaml_path.exists():
    print('[STEP 1] Creating COCO subset from validation set (70/20/10)...')
    
    # Load full COCO first
    print('  → Loading COCO full dataset...')
    full_coco_info = check_det_dataset('coco.yaml')
    
    # Get validation images
    print('  → Extracting validation images...')
    val_list_file = resolve_path(full_coco_info.get('path', '.'), full_coco_info['val'])
    val_images_list = get_images_from_source(val_list_file, full_coco_info.get('path', '.'))
    total_images = len(val_images_list)
    print(f'  → Total validation images: {total_images}')
    
    # Split: first 70% train, then from remaining 30% split into 67% val + 33% test
    print('  → Splitting into train/val/test (70/20/10)...')
    train_subset, temp = train_test_split(
        val_images_list, 
        test_size=0.30,
        random_state=42
    )
    val_subset, test_subset = train_test_split(
        temp,
        test_size=1/3,
        random_state=42
    )
    
    print(f'  → Train: {len(train_subset)} images ({100*len(train_subset)/total_images:.1f}%)')
    print(f'  → Val:   {len(val_subset)} images ({100*len(val_subset)/total_images:.1f}%)')
    print(f'  → Test:  {len(test_subset)} images ({100*len(test_subset)/total_images:.1f}%)')
    
    # Create folder structure
    print('  → Creating folder structure...')
    subset_root = Path(full_coco_info['path']) / 'coco_subset'
    subset_root.mkdir(exist_ok=True)
    
    dirs = create_subset_structure(subset_root, splits=['train', 'val', 'test'])
    
    # Get COCO labels path
    coco_path = Path(full_coco_info['path'])
    labels_root = coco_path / 'labels'
    val_split_folder = val_images_list[0].parent.name if val_images_list else 'val2017'
    src_label_dir = labels_root / val_split_folder
    
    # Copy all splits
    print('  → Copying subsets with labels...')
    copy_subset_with_labels(train_subset, src_label_dir, dirs['train']['images'], 
                           dirs['train']['labels'], split_name='Train')
    copy_subset_with_labels(val_subset, src_label_dir, dirs['val']['images'], 
                           dirs['val']['labels'], split_name='Val')
    copy_subset_with_labels(test_subset, src_label_dir, dirs['test']['images'], 
                           dirs['test']['labels'], split_name='Test')
    
    # Create YAML config (YOLO uses train/val, test is separate)
    print('  → Creating YAML config...')
    coco_subset_yaml_content = f"""# COCO Subset Dataset (from val set, 70/20/10 split)
path: {subset_root.resolve()}
train: train/images
val: val/images

nc: {len(full_coco_info['names'])}
names: {full_coco_info['names']}
"""
    
    with open(subset_yaml_path, 'w', encoding='utf-8') as f:
        f.write(coco_subset_yaml_content)
    
    print(f'\n✓ Subset created at: {subset_root.resolve()}')
    print(f'✓ YAML config created at: {subset_yaml_path.resolve()}')
    print(f'✓ Test split available at: {(subset_root / "test").resolve()}')
    data_info = full_coco_info
else:
    print(f'✓ Found existing {DATA_YAML}, loading...')
    data_info = check_det_dataset(str(subset_yaml_path))

print(f'\n[STEP 2] Loading dataset info...')
print(f'Dataset config: {DATA_YAML}')
print(f'Project dir: {PROJECT_DIR}')
print(f'Classes: {len(data_info["names"])}')

In [ ]:
print('[INFO] Subset dataset is ready. Labels already copied in previous step.')
print(f'Data info loaded: {DATA_YAML}')

## **3. Phân tích tập dữ liệu (EDA)**
Phần này thống kê quy mô dữ liệu và phân bố lớp để kiểm tra mức độ đa dạng cũng như tính mất cân bằng trong tập train.

In [ ]:
# Reload data_info to ensure it's from subset
if subset_yaml_path.exists():
    data_info = check_det_dataset(str(subset_yaml_path))
    print('[INFO] Loaded subset dataset info')

# Get paths for EDA analysis
subset_root = Path(data_info['path'])
splits = ['train', 'val', 'test']
split_dirs = {}

# Setup paths for all splits
for split in splits:
    images_dir = subset_root / split / 'images'
    labels_dir = subset_root / split / 'labels'
    images = sorted(
        list(images_dir.glob('*.jpg')) +
        list(images_dir.glob('*.jpeg')) +
        list(images_dir.glob('*.png'))
    ) if images_dir.exists() else []
    split_dirs[split] = {
        'images_dir': images_dir,
        'labels_dir': labels_dir,
        'images': images,
        'count': len(images)
    }

# Display summary
print(f'Dataset Summary:')
print(f'  Train: {split_dirs["train"]["count"]} images')
print(f'  Val:   {split_dirs["val"]["count"]} images')
print(f'  Test:  {split_dirs["test"]["count"]} images')
print(f'Total:  {sum(d["count"] for d in split_dirs.values())} images')
print(f'Classes: {len(data_info["names"])}')

# Use train split for EDA (as primary)
train_images = split_dirs['train']['images']
train_images_dir = split_dirs['train']['images_dir']
label_train_path = split_dirs['train']['labels_dir']

print(f'\nEDA will analyze TRAIN split ({len(train_images)} images)')

In [ ]:
def count_class_distribution(label_dir, class_names):
    counts = {i: 0 for i in range(len(class_names))}
    for lb in sorted(label_dir.glob('*.txt')):
        with open(lb, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_idx = int(float(parts[0]))
                if cls_idx in counts:
                    counts[cls_idx] += 1

    df = pd.DataFrame({
        'class_id': list(counts.keys()),
        'class_name': [class_names[i] for i in counts.keys()],
        'instances': list(counts.values())
    }).sort_values('instances', ascending=False)
    return df

class_names = data_info['names']
dist_df = count_class_distribution(label_train_path, class_names)
display(dist_df.head(15))

plt.figure(figsize=(12, 5))
sns.barplot(data=dist_df.head(15), x='instances', y='class_name', palette='viridis')
plt.title('Top 15 class xuất hiện nhiều nhất (train set)')
plt.xlabel('Số lượng instance')
plt.ylabel('Class')
plt.tight_layout()
plt.show()

Từ biểu đồ phân bố lớp, có thể nhận thấy tần suất xuất hiện giữa các lớp không đồng đều. Đây là đặc điểm phổ biến của COCO và có thể ảnh hưởng trực tiếp đến chất lượng phát hiện ở các lớp hiếm.

Trong quá trình phân tích kết quả, cần theo dõi thêm lỗi false negative/false positive theo từng nhóm lớp để có đánh giá đầy đủ hơn.

## **4. Cấu hình huấn luyện mô hình**
Thiết lập cấu hình chung cho các thí nghiệm nhằm đảm bảo tính công bằng khi so sánh YOLO11n và YOLO11s.

In [ ]:
TRAIN_ARGS = {
    'data': str(subset_yaml_path),
    'imgsz': 640,
    'epochs': 10,
    'batch': 16,
    'workers': 4,
    'project': str(PROJECT_DIR),
    'pretrained': True,
    'optimizer': 'auto',
    'seed': 42,
    'patience': 20,
    'cos_lr': True,
    'close_mosaic': 10,
    'save_period': 1,  # save every epoch
    'exist_ok': True
}
TRAIN_ARGS

## **5. Huấn luyện và so sánh hai biến thể YOLOv11**
Phần này đáp ứng yêu cầu cốt lõi của đề bài: **so sánh ít nhất hai phương pháp/biến thể** trên cùng một pipeline dữ liệu và cùng cấu hình đánh giá.

In [ ]:
experiments = {
    'yolo11n': 'yolo11n.pt',
    'yolo11s': 'yolo11s.pt',
}

train_runs = globals().get('train_runs', {})


def get_completed_epochs(exp_name):
    run_dir = PROJECT_DIR / f'{exp_name}_coco'
    results_csv = run_dir / 'results.csv'
    if not results_csv.exists():
        return 0

    try:
        df = pd.read_csv(results_csv)
        if df.empty:
            return 0
        if 'epoch' in df.columns:
            return int(df['epoch'].dropna().max()) + 1
        return int(len(df))
    except Exception as e:
        print(f'[WARN] Không đọc được {results_csv}: {e}')
        return 0


def get_train_status(exp_name):
    target_epochs = int(TRAIN_ARGS.get('epochs', 0))
    completed_epochs = get_completed_epochs(exp_name)
    done = target_epochs > 0 and completed_epochs >= target_epochs
    return done, completed_epochs, target_epochs


def train_or_resume(exp_name, ckpt_name):
    run_name = f'{exp_name}_coco'
    run_dir = PROJECT_DIR / run_name
    last_ckpt = run_dir / 'weights' / 'last.pt'
    best_ckpt = run_dir / 'weights' / 'best.pt'

    # Ultralytics validates `project` as a run folder name. Keep it relative and
    # anchor the current working directory to the Drive runs folder so outputs
    # still land in the right persistent location.
    os.chdir(str(PROJECT_DIR.parent))
    train_project = PROJECT_DIR.name
    train_name = run_name

    if last_ckpt.exists():
        print(f'[RESUME] {exp_name}: {last_ckpt}')
        model = YOLO(str(last_ckpt))
        return model.train(resume=True)

    print(f'[NEW RUN] {exp_name}: {ckpt_name}')
    model = YOLO(str(best_ckpt)) if best_ckpt.exists() else YOLO(ckpt_name)
    run_args = TRAIN_ARGS.copy()
    run_args['project'] = train_project
    run_args['name'] = train_name
    return model.train(**run_args)


train_done_flags = globals().get('train_done_flags', {})
for exp_name in experiments.keys():
    done, completed, target = get_train_status(exp_name)
    train_done_flags[exp_name] = done
    state = 'DONE' if done else 'PENDING'
    print(f'[{state}] {exp_name}: {completed}/{target} epochs')

In [ ]:
exp_name = 'yolo11n'
if train_done_flags.get(exp_name, False):
    done, completed, target = get_train_status(exp_name)
    print(f'[SKIP] {exp_name} đã train xong ({completed}/{target} epochs).')
else:
    ckpt = experiments[exp_name]
    result = train_or_resume(exp_name, ckpt)
    train_runs[exp_name] = result
    done, completed, target = get_train_status(exp_name)
    train_done_flags[exp_name] = done
    print(f'[STATUS] {exp_name}: {completed}/{target} epochs | done={done}')

print(train_done_flags)
train_runs

In [ ]:
exp_name = 'yolo11s'
if train_done_flags.get(exp_name, False):
    done, completed, target = get_train_status(exp_name)
    print(f'[SKIP] {exp_name} đã train xong ({completed}/{target} epochs).')
else:
    ckpt = experiments[exp_name]
    result = train_or_resume(exp_name, ckpt)
    train_runs[exp_name] = result
    done, completed, target = get_train_status(exp_name)
    train_done_flags[exp_name] = done
    print(f'[STATUS] {exp_name}: {completed}/{target} epochs | done={done}')

print(train_done_flags)
train_runs

## **6. Đánh giá mô hình trên tập validation**
Báo cáo các metric chuẩn cho bài toán object detection: Precision, Recall, mAP@0.5, mAP@0.5:0.95 và tốc độ suy luận.

In [ ]:
def get_run_info(exp_name):
    run_name = f'{exp_name}_coco'
    run_dir = PROJECT_DIR / run_name
    best_weight = run_dir / 'weights' / 'best.pt'
    last_weight = run_dir / 'weights' / 'last.pt'
    return {
        'model': exp_name,
        'run_dir': run_dir,
        'best_weight': best_weight,
        'last_weight': last_weight,
        'best_exists': best_weight.exists(),
        'last_exists': last_weight.exists(),
    }


def evaluate_run(exp_name, run_info):
    if not run_info['best_exists']:
        print(f"[SKIP] {exp_name}: chưa có best.pt tại {run_info['best_weight']}")
        return None

    model = YOLO(str(run_info['best_weight']))

    # Define the save directory for detection results
    # This will be REPORT_DIR / 'detections' / 'exp_name_coco'
    run_detections_dir = DETECTIONS_DIR / f'{exp_name}_coco'
    run_detections_dir.mkdir(parents=True, exist_ok=True)

    metrics = model.val(data=str(subset_yaml_path), split='val', save_json=True, plots=True, project=str(run_detections_dir.parent), name=run_detections_dir.name)
    return {
        'model': exp_name,
        'run_dir': str(run_info['run_dir']),
        'best_weight': str(run_info['best_weight']),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'inference_ms_per_image': float(metrics.speed.get('inference', float('nan'))),
        'val_dir': str(metrics.save_dir) # metrics.save_dir will now point to the correct path
    }


run_registry = {name: get_run_info(name) for name in experiments.keys()}
records = []

run_root = PROJECT_DIR
REPORT_DIR = run_root / 'report_artifacts'
FIG_DIR = REPORT_DIR / 'figures'
TABLE_DIR = REPORT_DIR / 'tables'
DETECTIONS_DIR = REPORT_DIR / 'detections' # Define detections directory
for d in [REPORT_DIR, FIG_DIR, TABLE_DIR, DETECTIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for name, info in run_registry.items():
    rec = evaluate_run(name, info)
    if rec is not None:
        records.append(rec)

result_df = pd.DataFrame(records)
if not result_df.empty:
    result_df = result_df.sort_values('mAP50-95', ascending=False).reset_index(drop=True)


metrics_json = TABLE_DIR / 'evaluation_records.json'
with open(metrics_json, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

display(result_df)
print(f'Đã tạo thư mục báo cáo: {REPORT_DIR.resolve()}')
print(f'Đã lưu metrics raw: {metrics_json.resolve()}')

In [ ]:
if result_df.empty:
    print('Chưa có run nào có best.pt để vẽ biểu đồ. Hãy train xong ít nhất 1 mô hình rồi chạy lại cell này.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    sns.barplot(data=result_df, x='model', y='mAP50', ax=axes[0], palette='Set2')
    axes[0].set_title('mAP@0.5')

    sns.barplot(data=result_df, x='model', y='mAP50-95', ax=axes[1], palette='Set2')
    axes[1].set_title('mAP@0.5:0.95')

    sns.barplot(data=result_df, x='model', y='inference_ms_per_image', ax=axes[2], palette='Set2')
    axes[2].set_title('Inference time (ms/image)')

    for ax in axes:
        ax.set_xlabel('Model')
        ax.tick_params(axis='x', rotation=0)

    plt.tight_layout()
    metrics_plot_path = FIG_DIR / 'metrics_comparison.png'
    fig.savefig(metrics_plot_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Đã lưu biểu đồ so sánh: {metrics_plot_path.resolve()}')

Nhìn vào bảng và biểu đồ, có thể đánh giá rõ trade-off giữa độ chính xác và tốc độ giữa hai biến thể YOLOv11.

Mô hình có mAP cao hơn chưa chắc là lựa chọn tốt nhất trong mọi bối cảnh; với các bài toán thời gian thực, độ trễ suy luận là yếu tố cần cân nhắc tương đương.

## **7. Trực quan kết quả dự đoán**
Minh họa detection trên một số ảnh validation để quan sát trực tiếp chất lượng localization và classification.

In [ ]:
# Make Section 7 runnable independently in fresh Colab sessions.
if 'DATA_YAML' not in globals():
    USE_COCO128_DEBUG = globals().get('USE_COCO128_DEBUG', False)
    DATA_YAML = 'coco128.yaml' if USE_COCO128_DEBUG else 'coco.yaml'

if 'PROJECT_DIR' not in globals():
    base_root = globals().get('DRIVE_ROOT', Path.cwd())
    PROJECT_DIR = Path(base_root) / 'runs' / 'assignment2_yolov11_coco'

if 'FIG_DIR' not in globals():
    FIG_DIR = PROJECT_DIR / 'report_artifacts' / 'figures'
    FIG_DIR.mkdir(parents=True, exist_ok=True)

if 'result_df' not in globals():
    fallback_experiments = globals().get('experiments', {'yolo11n': 'yolo11n.pt', 'yolo11s': 'yolo11s.pt'})
    rows = []
    for exp_name in fallback_experiments.keys():
        best_weight = PROJECT_DIR / f'{exp_name}_coco' / 'weights' / 'best.pt'
        if best_weight.exists():
            rows.append({'model': exp_name, 'best_weight': str(best_weight)})
    result_df = pd.DataFrame(rows)

if 'val_images' not in globals():
    local_data_info = data_info if 'data_info' in globals() else check_det_dataset(str(subset_yaml_path if 'subset_yaml_path' in globals() else DATA_YAML))
    if 'resolve_path' in globals():
        val_images_path = resolve_path(local_data_info.get('path', '.'), local_data_info['val'])
    else:
        p = Path(local_data_info['val'])
        val_images_path = p if p.is_absolute() else (Path(local_data_info.get('path', '.')) / p)

    if val_images_path.is_file() and val_images_path.suffix.lower() == '.txt':
        # COCO YAML thường trỏ tới file txt chứa danh sách ảnh thay vì thư mục ảnh.
        val_images = []
        with open(val_images_path, 'r', encoding='utf-8') as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                p = Path(s)
                if not p.is_absolute():
                    p = Path(local_data_info.get('path', '.')) / p
                if p.exists():
                    val_images.append(p)
        val_images = sorted(val_images)
    else:
        val_images = sorted(
            list(val_images_path.glob('*.jpg')) +
            list(val_images_path.glob('*.jpeg')) +
            list(val_images_path.glob('*.png'))
        )

if result_df.empty:
    print('Không có kết quả đánh giá để trực quan. Hãy chạy Section 6 sau khi đã có best.pt.')
elif 'best_weight' not in result_df.columns:
    print('Thiếu cột best_weight trong result_df. Hãy chạy lại Section 6 trước.')
elif len(val_images) == 0:
    print('Không tìm thấy ảnh validation để trực quan hóa.')
else:
    best_model_name = str(result_df.iloc[0]['model'])
    best_model_path = Path(result_df.iloc[0]['best_weight'])
    if not best_model_path.exists():
        print(f'Không tìm thấy best.pt tại {best_model_path}. Hãy train/evaluate lại Section 5-6.')
    else:
        best_model = YOLO(str(best_model_path))
        sample_images = random.sample(val_images, k=min(8, len(val_images)))
        pred_results = best_model.predict(source=[str(p) for p in sample_images], conf=0.25, save=False, verbose=False)

        pred_dir = FIG_DIR / 'predictions'
        pred_dir.mkdir(parents=True, exist_ok=True)

        fig, axes = plt.subplots(2, 4, figsize=(18, 9))
        axes = axes.flatten()
        for i, (ax, pred) in enumerate(zip(axes, pred_results), start=1):
            plotted = pred.plot()[:, :, ::-1]
            ax.imshow(plotted)
            ax.axis('off')
            plt.imsave(pred_dir / f'{best_model_name}_sample_{i:02d}.png', plotted)

        for i in range(len(pred_results), len(axes)):
            axes[i].axis('off')

        plt.suptitle(f'Visualization - {best_model_name}', fontsize=14)
        plt.tight_layout()
        grid_path = FIG_DIR / f'visualization_grid_{best_model_name}.png'
        fig.savefig(grid_path, dpi=200, bbox_inches='tight')
        plt.show()

        print(f'Đã lưu grid visualize: {grid_path.resolve()}')
        print(f'Đã lưu ảnh prediction riêng lẻ tại: {pred_dir.resolve()}')

Quan sát định tính giúp bổ sung cho các metric định lượng, đặc biệt trong các trường hợp mô hình phát hiện đúng nhưng confidence thấp hoặc nhầm giữa các lớp gần nhau.

Kết hợp hai góc nhìn định lượng và định tính sẽ giúp phần thảo luận kết quả trong báo cáo thuyết phục hơn.

## **8. Tổng hợp kết quả cho báo cáo**

In [ ]:
# Make Section 8 runnable independently (even if Section 6/7 was skipped).
if 'PROJECT_DIR' not in globals():
    base_root = globals().get('DRIVE_ROOT', Path.cwd())
    PROJECT_DIR = Path(base_root) / 'runs' / 'assignment2_yolov11_coco'
print("hihi:" ,PROJECT_DIR);
REPORT_DIR = globals().get('REPORT_DIR', PROJECT_DIR / 'report_artifacts')
TABLE_DIR = globals().get('TABLE_DIR', REPORT_DIR / 'tables')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if 'result_df' not in globals() or result_df is None:
    result_df = pd.DataFrame()

experiments = globals().get('experiments', {'yolo11n': 'yolo11n.pt', 'yolo11s': 'yolo11s.pt'})
TRAIN_ARGS = globals().get('TRAIN_ARGS', {})

if 'run_registry' not in globals() or run_registry is None:
    run_registry = {}
    for exp_name in experiments.keys():
        run_dir = PROJECT_DIR / f'{exp_name}_coco'
        best_weight = run_dir / 'weights' / 'best.pt'
        last_weight = run_dir / 'weights' / 'last.pt'
        run_registry[exp_name] = {
            'run_dir': run_dir,
            'best_exists': best_weight.exists(),
            'last_exists': last_weight.exists(),
        }

summary_cols = ['model', 'precision', 'recall', 'mAP50', 'mAP50-95', 'inference_ms_per_image']
if result_df.empty:
    summary_df = pd.DataFrame(columns=summary_cols)
else:
    # Reindex avoids KeyError when result_df only has partial columns (e.g. Section 7 fallback).
    summary_df = result_df.reindex(columns=summary_cols).copy()
    summary_df = summary_df.round({
        'precision': 4,
        'recall': 4,
        'mAP50': 4,
        'mAP50-95': 4,
        'inference_ms_per_image': 2
    })

summary_csv = TABLE_DIR / 'comparison_metrics.csv'
summary_json = TABLE_DIR / 'comparison_metrics.json'
run_meta_yaml = TABLE_DIR / 'run_metadata.yaml'

summary_df.to_csv(summary_csv, index=False)
summary_df.to_json(summary_json, orient='records', force_ascii=False, indent=2)

run_metadata = {
    'dataset_yaml': str(globals().get('subset_yaml_path', globals().get('DATA_YAML', 'coco.yaml'))),
    'experiments': experiments,
    'train_args': TRAIN_ARGS,
    'run_dirs': {k: str(v['run_dir']) for k, v in run_registry.items()},
    'best_weight_exists': {k: bool(v.get('best_exists', False)) for k, v in run_registry.items()},
    'last_weight_exists': {k: bool(v.get('last_exists', False)) for k, v in run_registry.items()}
}
with open(run_meta_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(run_metadata, f, sort_keys=False, allow_unicode=True)

display(summary_df)
print(f'Đã lưu bảng CSV: {summary_csv.resolve()}')
print(f'Đã lưu bảng JSON: {summary_json.resolve()}')
print(f'Đã lưu metadata run: {run_meta_yaml.resolve()}')

## **9. Hướng mở rộng**
- So sánh thêm với mô hình khác họ one-stage/two-stage (VD: YOLO11 vs Faster R-CNN/RT-DETR).
- Phân tích lỗi FN/FP theo từng lớp và confusion matrix.
- Đánh giá trade-off tốc độ/chính xác trên nhiều kích thước model (n/s/m/l).
- Demo real-time với webcam/video và triển khai giao diện Gradio/Streamlit.